# Self-Supervised Learning — Implementations

Two pretext tasks, revisited. InfoNCE gets the one-line torch identity — NT-Xent *is* cross-entropy over the similarity matrix — with autograd checked against the hand-derived gradient. MAE patch masking gets the same bookkeeping as tensor ops, kept bit-identical across lanes by drawing the permutation from NumPy in both.

## 22_simclr_info_nce

Pull two views of the same point together, push every other point away. Only `info_nce_loss` is the catalogued step (`SimpleMLP` is demo tooling). There is no separate library lane: `F.cross_entropy` over the masked similarity matrix is exactly how libraries express NT-Xent, so the torch lane below *is* the library expression — a separate lane would re-run the same line.

### torch

The scratch code builds the softmax and its gradient by hand; here the whole objective is one `F.cross_entropy` over the diagonal-masked similarity matrix, and `Z.grad` is `dZ`. **What torch adds:** the recognition that InfoNCE is a 2N-way classification problem — which is why autograd's gradient matches the derived formula to machine precision, and why no library ships a separate InfoNCE primitive.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. Stack both views: Z = cat([z1, z2]); row i's positive lives at (i + N) mod 2N.
# 2. masked_fill the diagonal with -inf so no row counts itself among the candidates.
# 3. F.cross_entropy(sim, labels) is NT-Xent: softmax then -log at the positive.
# 4. requires_grad_ on Z, then loss.backward(); Z.grad is the notebook's dZ.
# 5. The scratch adds 1e-8 inside its log: expect ~1e-8 loss delta, exact gradients.


def info_nce_loss(z1, z2, temperature=0.5):
    """NT-Xent written as cross-entropy over the masked similarity matrix.
    One F.cross_entropy call replaces the hand-built softmax, and one
    loss.backward() replaces the hand-derived gradient dZ."""
    z1t = torch.as_tensor(np.asarray(z1, dtype=float))
    z2t = torch.as_tensor(np.asarray(z2, dtype=float))
    N = z1t.shape[0]
    Z = torch.cat([z1t, z2t], dim=0).requires_grad_(True)

    sim = (Z @ Z.T) / temperature
    sim = sim.masked_fill(torch.eye(2 * N, dtype=torch.bool), float("-inf"))

    labels = torch.cat([torch.arange(N, 2 * N), torch.arange(N)])
    loss = F.cross_entropy(sim, labels)
    loss.backward()
    return float(loss), Z.grad.numpy()


In [ ]:
# exports: loss, dZ
_rng_eq = np.random.default_rng(2201)
_z1_eq = _rng_eq.normal(size=(6, 4))
_z1_eq = _z1_eq / np.linalg.norm(_z1_eq, axis=1, keepdims=True)
_z2_eq = _z1_eq + 0.1 * _rng_eq.normal(size=(6, 4))
_z2_eq = _z2_eq / np.linalg.norm(_z2_eq, axis=1, keepdims=True)

loss, dZ = info_nce_loss(_z1_eq, _z2_eq, temperature=0.5)
print("InfoNCE loss on 6 positive pairs:", round(loss, 6))


In [ ]:
# Autograd against the notebook's hand-derived gradient, recomputed here inline.
_Zc = np.vstack([_z1_eq, _z2_eq])
_S = _Zc @ _Zc.T / 0.5
np.fill_diagonal(_S, -np.inf)
_E = np.exp(_S - _S.max(axis=1, keepdims=True))
_P = _E / _E.sum(axis=1, keepdims=True)
_lab = np.concatenate([np.arange(6, 12), np.arange(6)])
_G = _P.copy()
_G[np.arange(12), _lab] -= 1.0
_G = _G / (12 * 0.5)
_dZ_hand = _G @ _Zc + _G.T @ _Zc
assert np.max(np.abs(dZ - _dZ_hand)) < 1e-12, "autograd reproduces the derived gradient"
assert loss > 0.0, "-log of a probability over 11 candidates is strictly positive"
_loss_swap, _dZ_swap = info_nce_loss(_z2_eq, _z1_eq, temperature=0.5)
assert abs(_loss_swap - loss) < 1e-12, "the loss is symmetric in the two views"
_loss_mis, _dZ_mis = info_nce_loss(_z1_eq, np.roll(_z2_eq, 1, axis=0), temperature=0.5)
assert _loss_mis > loss, "mispairing the views must raise the loss"


## 22_mae_patch_masking

Hide 75% of the image, remember where. Masking is index bookkeeping with nothing to differentiate, so this torch lane is not an autograd exercise — and there is no library lane, because no mainstream library exposes bare patch masking as a comparable estimator; it lives inside model pipelines.

### torch

The same bookkeeping on tensors: a NumPy permutation picks the kept patches, `repeat_interleave` inflates the patch mask to pixels, and boolean indexing zeroes the rest. **What torch adds:** the exact tensor ops a real MAE pipeline runs on the GPU — while drawing the permutation from NumPy keeps the mask bit-identical to the scratch lane's.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Keep the permutation in NumPy: rng.permutation is the notebook's randomness source.
# 2. Scatter True at the kept indices; ~keep is the notebook's boolean patch mask.
# 3. repeat_interleave along both axes is torch's np.kron with a block of ones.
# 4. clone() before zeroing - boolean-index assignment mutates the tensor in place.


def mask_image_patches(image, patch_size=4, mask_ratio=0.75):
    """MAE-style random patch masking on tensors. The permutation is still
    drawn from the notebook's NumPy rng, so with the same seed the mask is
    bit-identical to the scratch lane's."""
    img = torch.as_tensor(np.asarray(image, dtype=float))
    H, W = img.shape
    num_patches = (H // patch_size) * (W // patch_size)
    num_keep = int(num_patches * (1 - mask_ratio))
    indices = rng.permutation(num_patches)

    keep = torch.zeros(num_patches, dtype=torch.bool)
    keep[torch.as_tensor(indices[:num_keep])] = True
    mask_2d = (~keep).reshape(H // patch_size, W // patch_size)
    mask_pixel = mask_2d.repeat_interleave(patch_size, dim=0).repeat_interleave(patch_size, dim=1)

    masked_img = img.clone()
    masked_img[mask_pixel] = 0.0
    return masked_img


In [ ]:
# exports: masked_img, frac_masked
rng = np.random.default_rng(220)
_img_eq = np.arange(1.0, 257.0).reshape(16, 16)

_masked_t = mask_image_patches(_img_eq, patch_size=4, mask_ratio=0.75)
masked_img = _masked_t.numpy()
frac_masked = float((_masked_t == 0).to(torch.float64).mean())
print("masked fraction of pixels:", frac_masked)


In [ ]:
# The same seed must reproduce the scratch permutation - and therefore the mask.
_rng_chk = np.random.default_rng(220)
_idx_chk = _rng_chk.permutation(16)
_mask_chk = np.ones(16, dtype=bool)
_mask_chk[_idx_chk[:4]] = False
_px_chk = np.kron(_mask_chk.reshape(4, 4), np.ones((4, 4))).astype(bool)
assert np.array_equal(masked_img == 0, _px_chk), "same seed, same permutation, same mask"
assert abs(frac_masked - 0.75) < 1e-12, "exactly 12 of the 16 patches are zeroed"
assert np.array_equal(masked_img[~_px_chk], _img_eq[~_px_chk]), "kept pixels are untouched"
_sums = (masked_img == 0).reshape(4, 4, 4, 4).transpose(0, 2, 1, 3).reshape(16, 16).sum(axis=1)
assert np.all((_sums == 0) | (_sums == 16)), "masking is patch-aligned: whole 4x4 blocks or nothing"
